In [ ]:
from numba import config

#config.DISABLE_JIT = True

In [2]:
from tardis.io.configuration.config_reader import Configuration
from tardis.workflows.type_iip_workflow import TypeIIPWorkflow

Iterations:          0/? [00:00<?, ?it/s]

Packets:             0/? [00:00<?, ?it/s]

Initializing tabulator and plotly panel extensions for widgets to work


Auto-detected VSCode environment


Auto-detected VSCode environment


This notebook uses existing atomic data from the Vogl IIp paper converted to an updated format. It can be found on moria in /storage/group_storage/atom_data/iip_data/

Documentation builds render this notebook without executing it because it requires external atomic data and C-TARDIS reference files. Provide these files before running the notebook interactively.

In [3]:
iip_conf = Configuration.from_yaml('iip_config.yml')

The config here is not particularly good or appropriate for a IIp currently. It is mostly to test a pure hydrogen ejecta with continuum processes and non-thermal equilibrium plasma calculations. None of the other values are necessarily appropriate to model a IIp. 

# MUST BE AT LEAST 2 SHELLS

## Opacity and Monte Carlo thread counts

Set `plasma.continuum_interaction.nthreads` in `iip_config.yml` to control the maximum number of worker threads used by `ContinuumMacroAtomSolver`, including the absorbing Markov chain opacity solve. It must be a positive integer and defaults to `1`. This setting is independent of `montecarlo.nthreads`, which controls Monte Carlo transport threads.

For example, the following settings use one opacity worker and eight Monte Carlo threads:

```yaml
plasma:
  continuum_interaction:
    nthreads: 1
montecarlo:
  nthreads: 8
```

Consider a low thread count for the opacity solve: simulations containing many elements can require large amounts of memory per worker thread. Start with `1` and increase it only if sufficient memory is available. If this setting is omitted, opacity calculations use one worker regardless of the Monte Carlo thread count.

In [4]:
workflow = TypeIIPWorkflow(iip_conf)

BokehModel(combine_events=True, render_bundle={'docs_json': {'fc82c4e0-b61e-4584-9e34-ea3e98856e4d': {'version…

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [5]:
# import matplotlib.pyplot as plt
# import pandas as pd

# j_blues_ctardis = pd.read_csv(
#     "/home/afullard/tardis-chvogl-configs/j_blues_first.csv", index_col=0
# )
# plt.plot(workflow.plasma_solver.j_blues[0])
# plt.plot(j_blues_ctardis.values[0], linestyle="dashed")

In [6]:
workflow.run()

Shell No.,t_rad,next_t_rad,w,next_w
0,7.92e+03 K,7.98e+03 K,0.425,0.546
5,7.92e+03 K,7.93e+03 K,0.196,0.288


`xtol` termination condition is satisfied.
Function evaluations 18, initial cost 3.1116e-01, final cost 3.1116e-01, first-order optimality 2.98e+06.


RuntimeError: Thermal-balance solution failed final closure: level_residual=4.308e-04 exceeds 1.000e-10; electron_residual=7.975e-01 exceeds 2.000e-08; total_heating=2.540e-07 exceeds 5.000e-13; fractional_heating=9.891e-02 exceeds 2.000e-07

In [ ]:
import matplotlib.pyplot as plt
spectrum = workflow.spectrum_solver.spectrum_real_packets

In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 6.5))

spectrum.plot(label="Normal packets")

plt.xlim(500, 9000)
plt.title("TARDIS example model spectrum")
plt.xlabel(r"Wavelength [$\AA$]")
plt.ylabel(r"Luminosity density [erg/s/$\AA$]")
plt.legend()
plt.show()

In [ ]:
from tardis.visualization import SDECPlotter
import astropy.units as u

plotter = SDECPlotter.from_workflow(workflow)

plotter.generate_plot_mpl(packets_mode="real", packet_wvl_range=[500, 9000] * u.AA)

In [ ]:
workflow.transport_state.tracker_last_interaction_df[workflow.transport_state.tracker_last_interaction_df["last_interaction_type"] == "CONTINUUM_PROCESS"]

In [ ]:
from tardis.visualization.tools.liv_plot import LIVPlotter

In [ ]:
plotter = LIVPlotter.from_workflow(workflow)

plotter.generate_plot_mpl(packets_mode="real")